# Storage & Extraction - Scale and tune

Swap storage backends, do incremental inserts, tune extraction quality vs speed.

**Run:**
```bash
dotenvx run -- uv run jupyter nbconvert --to notebook --execute --inplace notebooks/04_storage_and_extraction.ipynb
```


In [1]:
import json
import logging
import sqlite3
import time
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

WORKING_DIR = Path("./_cache/04_storage_and_extraction")
WORKING_DIR.mkdir(parents=True, exist_ok=True)


## Imports + sample data

In [2]:
from nano_graphrag import GraphRAG, QueryParam
from nano_graphrag._storage import SQLiteGraphStorage, NetworkXStorage

TEXT = (
    "Alice works at Wonderland Labs as a quantum researcher. "
    "Bob is the CTO of Wonderland Labs and oversees the AI division. "
    "Carol founded DataForge, a startup building graph databases. "
    "Dave joined DataForge as lead engineer after leaving CloudCorp. "
    "Eve is a security consultant who audits both Wonderland Labs and DataForge."
)
print("Sample data ready (5 entities, 3 organizations)")


Sample data ready (5 entities, 3 organizations)


## Default backend: NetworkX

In [3]:
dir_nx = WORKING_DIR / "default"
dir_nx.mkdir(parents=True, exist_ok=True)
rag_nx = GraphRAG(working_dir=str(dir_nx), enable_llm_cache=True)

await rag_nx.ainsert(TEXT)
result = await rag_nx.aquery("Who works at Wonderland Labs?", QueryParam())
print(f"Query: {result[:120]}...")
print(f"Graph:  {rag_nx.graph_storage_cls.__name__}")
print(f"Vector: {rag_nx.vector_db_storage_cls.__name__}")
for f in sorted(dir_nx.iterdir()):
    sz = f.stat().st_size
    print(f"    {f.name:45s} {sz/1024:.1f} KB" if sz > 1024 else f"    {f.name:45s} {sz} B")


2026-05-17T12:09:20.963054Z [info     ] tokenizer_loading              [nano-graphrag] model_name=gpt-4o tokenizer_type=tiktoken


INFO:nano-graphrag:{'tokenizer_type': 'tiktoken', 'model_name': 'gpt-4o', 'event': 'tokenizer_loading', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:09:20.963054Z'}


2026-05-17T12:09:21.060484Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=llm_response_cache


INFO:nano-graphrag:{'namespace': 'llm_response_cache', 'entries': 0, 'event': 'sqlite_kv_loaded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:09:21.060484Z'}


2026-05-17T12:09:21.061038Z [info     ] litellm_configured             [nano-graphrag] api_base=None model=openrouter/google/gemma-4-31b-it structured_output=True


...

2026-05-17T12:10:52.552937Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=23 cost_usd=0.0 latency_ms=2358.4 mode=global model=openrouter/google/gemma-4-31b-it prompt_tokens=359 run_id=b9b5db26 total_tokens=382


INFO:nano-graphrag:{'model': 'openrouter/google/gemma-4-31b-it', 'latency_ms': 2358.4, 'prompt_tokens': 359, 'completion_tokens': 23, 'total_tokens': 382, 'cost_usd': 0.0, 'event': 'llm_call_complete', 'mode': 'global', 'run_id': 'b9b5db26', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:10:52.552937Z'}


2026-05-17T12:10:52.556783Z [info     ] query_complete                 [nano-graphrag] answer_chars=100 latency_ms=6588.3 mode=global run_id=b9b5db26


INFO:nano-graphrag:{'latency_ms': 6588.3, 'answer_chars': 100, 'event': 'query_complete', 'mode': 'global', 'run_id': 'b9b5db26', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:10:52.556783Z'}



Provider List: https://docs.litellm.ai/docs/providers

Query: Alice (a quantum researcher) and Bob (the CTO who oversees the AI division) work at Wonderland Labs....
Graph:  NetworkXStorage
Vector: HNSWVectorStorage
    entities_hnsw.index                           129.3 KB
    entities_hnsw_metadata.mpk                    964 B
    entity_registry.json                          3.6 KB
    graph_chunk_entity_relation.graphml           7.2 KB
    kv_store_community_reports.db                 4.0 KB
    kv_store_community_reports.db-shm             32.0 KB
    kv_store_community_reports.db-wal             60.4 KB
    kv_store_document_index.db                    4.0 KB
    kv_store_document_index.db-shm                32.0 KB
    kv_store_document_index.db-wal                36.2 KB
    kv_store_full_docs.db                         4.0 KB
    kv_store_full_docs.db-shm                     32.0 KB
    kv_store_full_docs.db-wal                     20.1 KB
    kv_store_graph_contribution_ind

## SQLite graph backend (crash-safe, atomic)

In [4]:
dir_sqlite = WORKING_DIR / "sqlite"
dir_sqlite.mkdir(parents=True, exist_ok=True)
rag_sqlite = GraphRAG(
    working_dir=str(dir_sqlite),
    graph_storage_cls=SQLiteGraphStorage,
    enable_llm_cache=True,
)

await rag_sqlite.ainsert(TEXT)

db_path = dir_sqlite / "graph_chunk_entity_relation.db"
conn = sqlite3.connect(str(db_path))
nodes = conn.execute("SELECT COUNT(*) FROM nodes").fetchone()[0]
edges = conn.execute("SELECT COUNT(*) FROM edges").fetchone()[0]
sample = conn.execute("SELECT id, data FROM nodes LIMIT 3").fetchall()
conn.close()

print(f"Nodes: {nodes}, Edges: {edges}")
for nid, ds in sample:
    d = json.loads(ds); print(f"  {d.get('entity_name','?'):20s} ({d.get('entity_type','?')})")


2026-05-17T12:10:52.565435Z [info     ] tokenizer_loading              [nano-graphrag] model_name=gpt-4o tokenizer_type=tiktoken


INFO:nano-graphrag:{'tokenizer_type': 'tiktoken', 'model_name': 'gpt-4o', 'event': 'tokenizer_loading', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:10:52.565435Z'}


2026-05-17T12:10:52.568589Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=llm_response_cache


INFO:nano-graphrag:{'namespace': 'llm_response_cache', 'entries': 0, 'event': 'sqlite_kv_loaded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:10:52.568589Z'}


2026-05-17T12:10:52.569136Z [info     ] litellm_configured             [nano-graphrag] api_base=None model=openrouter/google/gemma-4-31b-it structured_output=True


...

INFO:nano-graphrag:{'model': 'openrouter/google/gemma-4-31b-it', 'latency_ms': 6394.5, 'prompt_tokens': 1407, 'completion_tokens': 134, 'total_tokens': 1541, 'cost_usd': 0.0, 'event': 'llm_call_complete', 'run_id': 'd4b534ec', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:12:14.791116Z'}



Provider List: https://docs.litellm.ai/docs/providers



2026-05-17T12:12:21.239158Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=303 cost_usd=0.0 latency_ms=12841.3 model=openrouter/google/gemma-4-31b-it prompt_tokens=1460 run_id=d4b534ec total_tokens=1763


INFO:nano-graphrag:{'model': 'openrouter/google/gemma-4-31b-it', 'latency_ms': 12841.3, 'prompt_tokens': 1460, 'completion_tokens': 303, 'total_tokens': 1763, 'cost_usd': 0.0, 'event': 'llm_call_complete', 'run_id': 'd4b534ec', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:12:21.239158Z'}



Provider List: https://docs.litellm.ai/docs/providers

Nodes: 8, Edges: 8
  BOB                  (PERSON)
  DATAFORGE            (ORGANIZATION)
  EVE                  (PERSON)


## Backend comparison table

In [5]:

print("+--------------------+---------------+------------------+---------------------------+")
print("| Backend            | Storage       | Execution        | Best for                  |")
print("+--------------------+---------------+------------------+---------------------------+")
print("| NetworkXStorage    | GraphML XML   | In-memory        | Dev, small graphs         |")
print("| SQLiteGraphStorage | SQLite DB     | Disk (WAL)       | Medium, crash-safe        |")
print("| Neo4jStorage       | Neo4j server  | External server  | Production, large graphs  |")
print("+--------------------+---------------+------------------+---------------------------+")
print()
print("Vector backends:")
print("  HNSWVectorStorage    - fast ANN, default")
print("  NanoVectorDBStorage  - pure Python, no C deps")
print()
print("Swap:")
print("  rag = GraphRAG(graph_storage_cls=SQLiteGraphStorage)")
print("  rag = GraphRAG(vector_db_storage_cls=NanoVectorDBStorage)")


+--------------------+---------------+------------------+---------------------------+
| Backend            | Storage       | Execution        | Best for                  |
+--------------------+---------------+------------------+---------------------------+
| NetworkXStorage    | GraphML XML   | In-memory        | Dev, small graphs         |
| SQLiteGraphStorage | SQLite DB     | Disk (WAL)       | Medium, crash-safe        |
| Neo4jStorage       | Neo4j server  | External server  | Production, large graphs  |
+--------------------+---------------+------------------+---------------------------+

Vector backends:
  HNSWVectorStorage    - fast ANN, default
  NanoVectorDBStorage  - pure Python, no C deps

Swap:
  rag = GraphRAG(graph_storage_cls=SQLiteGraphStorage)
  rag = GraphRAG(vector_db_storage_cls=NanoVectorDBStorage)


## Incremental insertion

In [6]:
# ainsert_documents adds new content without reprocessing old docs.
# Detection is by content hash - no .db "cursor" to track.

dir_inc = WORKING_DIR / "incremental"
dir_inc.mkdir(parents=True, exist_ok=True)
rag_inc = GraphRAG(working_dir=str(dir_inc), enable_llm_cache=True)

docs_round1 = {"doc_a": "The Eiffel Tower is located in Paris, France."}
docs_round2 = {"doc_b": "The Louvre Museum is also in Paris, near the Seine River."}

t0 = time.time()
await rag_inc.ainsert_documents(docs_round1)
print(f"Round 1: {time.time()-t0:.1f}s")

t0 = time.time()
await rag_inc.ainsert_documents(docs_round2)
print(f"Round 2: {time.time()-t0:.1f}s (only doc_b processed)")

result = await rag_inc.aquery("What is in Paris?", QueryParam())
print(f"Query: {result}")


2026-05-17T12:12:21.258689Z [info     ] tokenizer_loading              [nano-graphrag] model_name=gpt-4o tokenizer_type=tiktoken


INFO:nano-graphrag:{'tokenizer_type': 'tiktoken', 'model_name': 'gpt-4o', 'event': 'tokenizer_loading', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:12:21.258689Z'}


2026-05-17T12:12:21.260725Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=llm_response_cache


INFO:nano-graphrag:{'namespace': 'llm_response_cache', 'entries': 0, 'event': 'sqlite_kv_loaded', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:12:21.260725Z'}


2026-05-17T12:12:21.261234Z [info     ] litellm_configured             [nano-graphrag] api_base=None model=openrouter/google/gemma-4-31b-it structured_output=True


...

2026-05-17T12:13:33.433473Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=17 cost_usd=0.0 latency_ms=2045.6 mode=global model=openrouter/google/gemma-4-31b-it prompt_tokens=283 run_id=f22cdc26 total_tokens=300


INFO:nano-graphrag:{'model': 'openrouter/google/gemma-4-31b-it', 'latency_ms': 2045.6, 'prompt_tokens': 283, 'completion_tokens': 17, 'total_tokens': 300, 'cost_usd': 0.0, 'event': 'llm_call_complete', 'mode': 'global', 'run_id': 'f22cdc26', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:13:33.433473Z'}


2026-05-17T12:13:33.437432Z [info     ] query_complete                 [nano-graphrag] answer_chars=68 latency_ms=3343.3 mode=global run_id=f22cdc26


INFO:nano-graphrag:{'latency_ms': 3343.3, 'answer_chars': 68, 'event': 'query_complete', 'mode': 'global', 'run_id': 'f22cdc26', 'logger': 'nano-graphrag', 'level': 'info', 'timestamp': '2026-05-17T12:13:33.437432Z'}



Provider List: https://docs.litellm.ai/docs/providers

Query: The Eiffel Tower, a wrought-iron lattice tower, is located in Paris.


## Extraction tuning knobs

In [7]:

print("Key extraction parameters (set via GraphRAG kwargs):")
print()
print("  chunk_size=1200              - chars per chunk")
print("  chunk_overlap=200            - overlap between chunks")
print("  doc_extraction_max_async=4   - parallel docs processed")
print("  extraction_max_async=16      - parallel chunk extractions per doc")
print()
print("  entity_extraction_max_gleaning=1  - LLM re-reads chunks")
print("  enable_entity_link=True           - link entity variants")
print()
print("  best_model / embedding_model      - which model to use")
print()
print("Tuning tips:")
print("  - Higher chunk_size = fewer LLM calls but coarser extraction")
print("  - Higher max_async = faster but more API rate-limit risk")
print("  - More gleaning = catches more entities but costs more")
print("  - entity_link=True helps with entity resolution")


Key extraction parameters (set via GraphRAG kwargs):

  chunk_size=1200              - chars per chunk
  chunk_overlap=200            - overlap between chunks
  doc_extraction_max_async=4   - parallel docs processed
  extraction_max_async=16      - parallel chunk extractions per doc

  entity_extraction_max_gleaning=1  - LLM re-reads chunks
  enable_entity_link=True           - link entity variants

  best_model / embedding_model      - which model to use

Tuning tips:
  - Higher chunk_size = fewer LLM calls but coarser extraction
  - Higher max_async = faster but more API rate-limit risk
  - More gleaning = catches more entities but costs more
  - entity_link=True helps with entity resolution
